# Subaward leverage review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** A2 — DIB integration via subaward links; A3 — follow-on leverage  
framing ([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/nano_subaward_leverage.py` (inputs from  
`nano_ws5a_subawards.py`)
**Data as of:** the WS5a subaward snapshot recorded by the generating scripts  

Companion view over the leverage artifact, focused on denominator definitions,
dominant-prime concentration, and firm-level distributions. Exploratory-tier and
non-citable.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

## Data contract

- **Population:** WS5a strong-tier firms only — firms whose subaward match already
  cleared the strong evidence bar. Not the full cohort.
- **Grain:** firm.
- **Keys:** normalized firm name (via `sbir_etl.identity` normalization inside the
  script); `company` is the display label.
- **Denominator:** `cumulative_sbir_usd` is the firm's TOTAL SBIR history (all phases,
  all years, all areas) — deliberately matching Finding 2's methodology, not the area
  cohort's award dollars. A leverage ratio quoted against a different denominator is a
  different number.
- **Caveats:** subaward streams are recurring revenue, not an exit price; the script's
  own docstring warns against reading this as acquisition-style leverage.
  `known_finding2_acquisition` firms enrich an already-known story.

In [ ]:
ARTIFACTS = {
    "subaward leverage": REPORT_DIR / "subaward_leverage.csv",
    "WS5a subawards": REPORT_DIR / "ws5a_subawards.csv",
}
GENERATORS = {
    "subaward leverage": "scripts/data/nano_subaward_leverage.py",
    "WS5a subawards": "scripts/data/nano_ws5a_subawards.py",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

def load_artifact(name: str) -> pd.DataFrame:
    """Read a canonical CSV artifact, or return an empty frame with a hint."""
    path = ARTIFACTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first."
        )
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

## Firm distributions

Leverage and dollar distributions, split by whether the firm is already a known
Finding-2 acquisition. Medians and percentiles, never bare means — the top firms
dominate the raw sum.

In [ ]:
leverage = load_artifact("subaward leverage")
if leverage.empty:
    distribution = pd.DataFrame()
else:
    leverage["leverage"] = pd.to_numeric(leverage["leverage"], errors="coerce")
    distribution = (
        leverage.groupby("known_finding2_acquisition")
        .agg(
            firms=("company", "size"),
            median_sub_usd=("post_award_subaward_usd", "median"),
            p90_sub_usd=("post_award_subaward_usd", lambda s: s.quantile(0.9)),
            total_sub_usd=("post_award_subaward_usd", "sum"),
            median_leverage=("leverage", "median"),
            share_over_1x=("leverage", lambda s: (s >= 1).mean()),
        )
    )
distribution

## Dominant-prime sensitivity

How much of each firm's subaward volume rides on a single prime relationship, and how
much of the population total sits in the top firms. A distribution whose total is
driven by a handful of large going concerns supports a different sentence than a broad
base of leverage ≥ 1 firms.

In [ ]:
if leverage.empty:
    concentration = pd.DataFrame()
else:
    ranked = leverage.sort_values("post_award_subaward_usd", ascending=False).reset_index(drop=True)
    total = ranked["post_award_subaward_usd"].sum()
    concentration = pd.DataFrame({
        "top_n": [1, 5, 10, 25],
        "share_of_total_sub_usd": [
            round(ranked["post_award_subaward_usd"].head(n).sum() / total, 3) for n in (1, 5, 10, 25)
        ],
    })
    print("Top firms with their primes (top_primes is the script's ranked prime list):")
    display(ranked.loc[:9, ["company", "post_award_subaward_usd", "leverage", "top_primes"]])
concentration

## Alternative denominators

The artifact carries one denominator (total SBIR history). Alternatives — area-cohort
SBIR dollars only, Phase II dollars only — need `data/raw/sbir/award_data.csv` and a
rerun of the canonical script with a changed definition, not a notebook re-join.
Record which denominator any quoted ratio used.

## Interpretation log

| Observation | Denominator used | Concentration caveat | Defensible statement |
|---|---|---|---|
| _Draft_ | _Total-SBIR vs alternative_ | _Top-firm share_ | _Recurring revenue, not exit leverage_ |